# 하이브리드 RAG 기반 chatbot 구현

## 환경 설정

In [1]:
# 필요한 모듈 / 클래스 / 함수 임포트
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from mecab import MeCab

C:\Users\playdata2\AppData\Local\Temp\ipykernel_16560\1013962079.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# .env file -> API Key 가져오기
env_file_path='.env'
output = load_dotenv(env_file_path)
print(output)

True


## 데이터 로드 및 변환

In [3]:
# 데이터 로드 및 변환 도구 생성
loader = TextLoader(file_path="tech_support_manual.txt", encoding="utf-8")

# load 함수 -> Document 자료형 생성
documents = loader.load()
print(f'documents: {documents}')

documents: [Document(metadata={'source': 'tech_support_manual.txt'}, page_content='질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?\n답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.\n\n질문: 스마트 조명 스위치 퓨어(Pure) v2 모델에서 빨간색 LED가 3번 뻐끔거리며 깜빡이고 연결이 끊겨요.\n답변: LED 빨간색 3회 점등은 Wi-Fi 인증 실패 및 공유기 신호 세기 약화 증상입니다. 스마트폰 앱의 [설정] > [네트워크] > [신호 재연결] 메뉴로 이동하여 2.4GHz 대역의 Wi-Fi에 다시 연결해 주세요. 5GHz 대역은 지원하지 않습니다.\n\n질문: 도어락 모델 DL-900 시리즈의 비상 비밀번호 초기화 규정(제14조 2항)은 어떻게 되나요?\n답변: 도어락 비상 초기화 규정 제14조 2항에 따라, 본인 확인을 위해 실물 신분증과 매뉴얼 후면의 시리얼 번호(S/N) 카드를 지참하시고 지정 대리점에 방문하셔야 합니다. 보안상 전화나 앱을 통한 원격 초기화는 불가능합니다.\n\n질문: 앱에서 네트워크 연결 안 됨 상태로 표시되고 코드 NET-104가 뜹니다.\n답변: NET-104 오류는 방화벽 또는 포트 차단 문제입니다. 사용 중인 공유기의 8080 및 443 포트 포워딩 상태를 확인하시고, 공유기 재부팅 후 앱을 다시 실행해 주세요.\n\n질문: 기기가 완전히 먹통이 되고 화면이 아무 반응도 없을 때 강제 재부팅하는 방법이 있나요?\n답변: 기기 측면의 음량 줄임(-) 버튼과 전원 버튼을 동시에 7초 이상 길게 누르면 강제 하드 리셋이 진행됩니다. 화면에 로고가 떠오를 때까지 버튼을 떼지 마세요.\n\n질문: 스

## 텍스트 분할

In [7]:
# 적절한 글자수(chunk_size)를 찾기 위한 계산

## 데이터 상태: 각 질문-답변 쌍 -> 문단('\n\n')으로 구분

## 파이썬으로 .txt 파일 읽기, 문자열로 변환

### 파일 경로 설정
file_path="tech_support_manual.txt"

### with open 구문, 문자열로 변환
with open(file_path, "r", encoding="utf-8") as file:
    manual_text = file.read()

## 문자열 -> split() -> 질문-답변 쌍으로 분할
qa_pairs = manual_text.split("\n\n")
print(qa_pairs)

print('-'*80)

## 각 질문-답변 쌍 별 글자수 계산

### 결과 저장 리스트 생성
character_length=[]

### for문 + len 함수 + append 함수
for pair in qa_pairs:
    character_length.append(len(pair))

print(character_length)

print('-'*80)

## 질문-답변 쌍 중 가장 글자 수가 긴 경우의 길이 추출
max_char_length = max(character_length)
print(f"가장 긴 Q&A 단락의 글자 수: {max_char_length}자")

print('-'*80)

## optimal_chunk_size 설정
optimal_chunk_size = 210

['질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?\n답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.', '질문: 스마트 조명 스위치 퓨어(Pure) v2 모델에서 빨간색 LED가 3번 뻐끔거리며 깜빡이고 연결이 끊겨요.\n답변: LED 빨간색 3회 점등은 Wi-Fi 인증 실패 및 공유기 신호 세기 약화 증상입니다. 스마트폰 앱의 [설정] > [네트워크] > [신호 재연결] 메뉴로 이동하여 2.4GHz 대역의 Wi-Fi에 다시 연결해 주세요. 5GHz 대역은 지원하지 않습니다.', '질문: 도어락 모델 DL-900 시리즈의 비상 비밀번호 초기화 규정(제14조 2항)은 어떻게 되나요?\n답변: 도어락 비상 초기화 규정 제14조 2항에 따라, 본인 확인을 위해 실물 신분증과 매뉴얼 후면의 시리얼 번호(S/N) 카드를 지참하시고 지정 대리점에 방문하셔야 합니다. 보안상 전화나 앱을 통한 원격 초기화는 불가능합니다.', '질문: 앱에서 네트워크 연결 안 됨 상태로 표시되고 코드 NET-104가 뜹니다.\n답변: NET-104 오류는 방화벽 또는 포트 차단 문제입니다. 사용 중인 공유기의 8080 및 443 포트 포워딩 상태를 확인하시고, 공유기 재부팅 후 앱을 다시 실행해 주세요.', '질문: 기기가 완전히 먹통이 되고 화면이 아무 반응도 없을 때 강제 재부팅하는 방법이 있나요?\n답변: 기기 측면의 음량 줄임(-) 버튼과 전원 버튼을 동시에 7초 이상 길게 누르면 강제 하드 리셋이 진행됩니다. 화면에 로고가 떠오를 때까지 버튼을 떼지 마세요.', '질문: 스마트 카메라 CAM-PRO-300의 SD 카드 인식 불가 오류 SD-301 조치법은 무엇인가요?\n답변: SD-301은 파일 시스템 호환성 오류입

In [8]:
# RecursiveCharacterTextSplitter 클래스 호출, 분할 도구 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=optimal_chunk_size,
    chunk_overlap=0
)

# 전용 함수: split_documents() 함수 호출 -> 분할된 문서 생성
docs = text_splitter.split_documents(documents)
print(f'분할된 원본 문서: \n{docs}')
print('-'*80)
print(f'분할된 문서 조각의 수: {len(docs)}')
print('-'*80)
print(f'분할된 문서 조각 예시: \n{docs[0]}')

분할된 원본 문서: 
[Document(metadata={'source': 'tech_support_manual.txt'}, page_content='질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?\n답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.'), Document(metadata={'source': 'tech_support_manual.txt'}, page_content='질문: 스마트 조명 스위치 퓨어(Pure) v2 모델에서 빨간색 LED가 3번 뻐끔거리며 깜빡이고 연결이 끊겨요.\n답변: LED 빨간색 3회 점등은 Wi-Fi 인증 실패 및 공유기 신호 세기 약화 증상입니다. 스마트폰 앱의 [설정] > [네트워크] > [신호 재연결] 메뉴로 이동하여 2.4GHz 대역의 Wi-Fi에 다시 연결해 주세요. 5GHz 대역은 지원하지 않습니다.'), Document(metadata={'source': 'tech_support_manual.txt'}, page_content='질문: 도어락 모델 DL-900 시리즈의 비상 비밀번호 초기화 규정(제14조 2항)은 어떻게 되나요?\n답변: 도어락 비상 초기화 규정 제14조 2항에 따라, 본인 확인을 위해 실물 신분증과 매뉴얼 후면의 시리얼 번호(S/N) 카드를 지참하시고 지정 대리점에 방문하셔야 합니다. 보안상 전화나 앱을 통한 원격 초기화는 불가능합니다.'), Document(metadata={'source': 'tech_support_manual.txt'}, page_content='질문: 앱에서 네트워크 연결 안 됨 상태로 표시되고 코드 NET-104가 뜹니다.\n답변: NET-104 오류는 방화벽 또는 포트 차단 문제입니다. 사용 중인 공유기

## MeCab 형태소 분석기 기반 토큰화 함수 정의 

In [9]:
# 형태소 분석 도구 생성
mecab = MeCab()

In [10]:
# 토큰화 함수 정의

def mecab_tokenizer(text):
    
    ## 의미 있는 형태소(명사, 용언, 숫자, 영어 및 특수 기호 등) 추출 결과 저장용 빈 리스트 생성
    tokens = []

    ## 품사 태깅 -> [('형태소', '품사'), ...]
    tags = mecab.pos(text)

    ## for문 정의
    for morph, pos in tags:
        if pos.startswith(('N', 'V', 'S')):
            tokens.append(morph)
    return tokens

## BM25 희소 검색기 생성

In [11]:
# BM25Retriever 클래스 호출, from_documents 전용 함수 호출 -> 검색기 생성
sparse_retriever = BM25Retriever.from_documents(
    documents=docs,
    preprocess_func=mecab_tokenizer,
    # 연관성이 높은 문서 추출 조건 설정
    k=2
) 

In [12]:
# BM25 희소 검색기 테스트

## 사용자 질의 설정
test_query = "ERR-502 오류코드 조치법"

## invoke() 함수 호출 -> 검색 실행 -> 결과: [Document1, Document2] 
retrieved_docs = sparse_retriever.invoke(test_query)

## for문 -> 각 Document -> page_content -> 텍스트 추출
for doc in retrieved_docs:
    print(f"내용: {doc.page_content}")
    print('-'*80)

내용: 질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?
답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.
--------------------------------------------------------------------------------
내용: 질문: 스마트 카메라 CAM-PRO-300의 SD 카드 인식 불가 오류 SD-301 조치법은 무엇인가요?
답변: SD-301은 파일 시스템 호환성 오류입니다. PC에서 SD 카드를 FAT32 포맷으로 재설정한 후 기기에 재장착하거나, 앱 내 [장치 관리] > [저장소 포맷] 기능을 이용해 초기화해 주세요.
--------------------------------------------------------------------------------


## ChromaDB 기반 밀집 검색기 생성

### 임베딩 모델 생성

In [13]:
# OpenAI -> 임베딩 모델

## 모델 이름 설정
model_name = "text-embedding-3-small"

## OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

### 백터 저장소 및 검색 도구 생성

In [14]:
# 백터 저장소 경로 설정
chroma_path = 'chroma_db_tech_support'

# 기존 DB가 존재하면 로드, 없으면 생성하여 저장
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

신규 벡터 저장소를 생성하고 저장했습니다.


In [25]:
# 검색 도구 생성
dense_retriever = vectorstore.as_retriever(search_kwargs={'k':2})
print(f'밀집 검색 도구: {dense_retriever}')

밀집 검색 도구: tags=['Chroma', 'OpenAIEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000013638F3E1E0> search_kwargs={'k': 2}


## [Hybrid] EnsembleRetriever를 이용한 검색기 결합 

In [27]:
# 내용 정리
"""
# EnsembleRetriever 클래스
1. 여러 개의 검색기(Retriever)가 반환한 문서 결과를 하나로 통합하고,
   가중치 기반의 상호 순위 융합(RRF) 알고리즘으로 순위를 재조정해 반환하는클래스
2. 핵심 매개변수
   1) retrievers: 결합할 검색기 리스트
   2) weights: 각 검색기의 결과에 부여할 가중치 리스트(예시: [0.5, 0.5])
"""

# EnsembleRetriever 클래스 호출, 하이브리드 검색 도구 생성
hybrid_retriever = EnsembleRetriever(
    retrievers=[sparse_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

print(f"하이브리드 검색 도구: {hybrid_retriever}")

하이브리드 검색 도구: retrievers=[BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001363AD08530>, k=2, preprocess_func=<function mecab_tokenizer at 0x000001363B3049A0>), VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000013638F3E1E0>, search_kwargs={'k': 2})] weights=[0.5, 0.5]


## RAG 체인 구성

### 프롬프트 생성

In [17]:
# 시스템 프롬프트 생성
system_prompt = """
당신은 스마트홈 IT 기기 및 지원 서비스의 전문 기술 지원 AI 엔지니어입니다.
제공된 [참고 컨텍스트] 정보를 바탕으로 고객의 제품 문의, 오류 코드 해결법, A/S 규정 질문에 정중하고 명확하게 답변해주세요.

[답변 작성 수칙]
1. [참고 컨텍스트]에 명시된 모델명, 오류 코드, 조치 단계, 규정 조항을 정확하게 인용하세요.
2. 답변을 시작할 때 '답변:'이라는 단어를 사용하지 마세요.
3. [참고 컨텍스트]에 고객 질문에 관한 내용이 없다면, "죄송합니다. 문의하신 내용에 대한 기술 매뉴얼 정보를 찾을 수 없습니다. 고객센터(1588-0000)로 문의 부탁드립니다."라고 답변하세요.

[참고 컨텍스트]
{context}
"""

# 사용자 프롬프트 생성
user_prompt="{question}"

# 전체 프롬프트 생성
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n당신은 스마트홈 IT 기기 및 지원 서비스의 전문 기술 지원 AI 엔지니어입니다.\n제공된 [참고 컨텍스트] 정보를 바탕으로 고객의 제품 문의, 오류 코드 해결법, A/S 규정 질문에 정중하고 명확하게 답변해주세요.\n\n[답변 작성 수칙]\n1. [참고 컨텍스트]에 명시된 모델명, 오류 코드, 조치 단계, 규정 조항을 정확하게 인용하세요.\n2. 답변을 시작할 때 \'답변:\'이라는 단어를 사용하지 마세요.\n3. [참고 컨텍스트]에 고객 질문에 관한 내용이 없다면, "죄송합니다. 문의하신 내용에 대한 기술 매뉴얼 정보를 찾을 수 없습니다. 고객센터(1588-0000)로 문의 부탁드립니다."라고 답변하세요.\n\n[참고 컨텍스트]\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


### LLM 생성

In [18]:
# ChatOpenAI 클래스 호출, llm 생성
llm = ChatOpenAI(model='gpt-5-nano')

### output parser 생성

In [19]:
# LLM 답변 -> AIMessage -> content="text" -> 기능: AIMessage.content
output_parser = StrOutputParser()

### rag chain 생성

In [20]:
# LCEL 방식 -> "|" 사용
rag_chain = prompt | llm | output_parser

## Chatbot 실행

In [28]:
# while 반복문 정의 및 실행

print("안녕하세요! 하이브리드 RAG 기반 고객 응대 챗봇입니다. 궁금한 점을 질문해주세요. (종료하려면 'exit' 입력)")

print('-'*80)

while True:
    try:
        ## 사용자 입력(질문) 생성
        user_question = input("사용자 질문: ")

        ## 사용자 입력 -> 종료 조건 만족 -> 종료
        if user_question.lower() == 'exit':
            print("챗봇을 종료합니다. 이용해주셔서 감사합니다.")
            break

        ## 하이브리드 검색 실행 (sparse 2개 + dense 2개 -> 결과 -> RRF 적용 -> 문서별 순위)
        retrieved_docs = hybrid_retriever.invoke(user_question)
        print(f'하이브리드 검색 결과: 총 {len(retrieved_docs)}개 문서 수집')

        print('-'*80)

        ## 하이브리드 검색 실행 -> 추출된 문서 리스트 -> 문자열 추출 -> 병합: "\n\n".join([text1, text2, ...]) 

        ### 추출한 텍스트를 담을 빈 리스트 생성
        contents = []

        ### 검색된 문서 리스트를 순회하며 텍스트(page_content) 추출
        for doc in retrieved_docs:
            text=doc.page_content
            contents.append(text) 

        ### 리스트(contents)의 모든 문자열 -> join() 함수 -> "\n\n" 기준으로 연결
        context_text = "\n\n".join(contents)
        print(f'답변의 근거: \n{context_text}')

        print('-'*80)

        ## rag_chain.invoke()를 호출, 사용자 질문에 대한 답변 생성
        response = rag_chain.invoke({
            "context": context_text,
            "question": user_question
        })

        ## 결과 출력
        print(f"챗봇 응답: \n{response}")
        print('='*80)

    except Exception as e:
        print(f'오류가 발생했습니다: {e}')

안녕하세요! 하이브리드 RAG 기반 고객 응대 챗봇입니다. 궁금한 점을 질문해주세요. (종료하려면 'exit' 입력)
--------------------------------------------------------------------------------
하이브리드 검색 결과: 총 3개 문서 수집
--------------------------------------------------------------------------------
답변의 근거: 
질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?
답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.

질문: 스마트 카메라 CAM-PRO-300의 SD 카드 인식 불가 오류 SD-301 조치법은 무엇인가요?
답변: SD-301은 파일 시스템 호환성 오류입니다. PC에서 SD 카드를 FAT32 포맷으로 재설정한 후 기기에 재장착하거나, 앱 내 [장치 관리] > [저장소 포맷] 기능을 이용해 초기화해 주세요.

질문: 앱에서 네트워크 연결 안 됨 상태로 표시되고 코드 NET-104가 뜹니다.
답변: NET-104 오류는 방화벽 또는 포트 차단 문제입니다. 사용 중인 공유기의 8080 및 443 포트 포워딩 상태를 확인하시고, 공유기 재부팅 후 앱을 다시 실행해 주세요.
--------------------------------------------------------------------------------
챗봇 응답: 
스마트 허브 SH-M200X의 ERR-502 전원 오류에 대한 조치법은 다음과 같습니다.

- ERR-502는 어댑터 전압 불균형 오류입니다.
- 1) 제품 후면의 리셋(Reset) 버튼을 10초간 누릅니다.
- 2) 제공